# 01. Collecting the Hotline data

I wanted to start this project with a consistent state-level measure of reported human trafficking activity. The National Human Trafficking Hotline publishes annual counts of signals received and trafficking cases identified by state.

I'm using 2024 as the starting year because it gives me a complete annual dataset that I can later compare with 2024 population and demographic data.

A major limitation is important from the beginning: Hotline data reflects situations that were reported to the Hotline. It does not represent the actual prevalence of human trafficking in a state. Reporting awareness, access to the Hotline, population size, outreach, and other factors can all affect these numbers.

**Source:** National Human Trafficking Hotline  
**Year used:** 2024

In [2]:
import pandas as pd
import requests

from bs4 import BeautifulSoup
from io import StringIO
from pathlib import Path


# Find the main project folder no matter where the notebook is launched from.
project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent


raw_data_dir = project_dir / "data" / "raw" / "hotline"
raw_data_dir.mkdir(parents=True, exist_ok=True)


print(project_dir)

c:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis


## Pulling the 2024 state table

The Hotline keeps several years of state statistics on the same webpage, so instead of assuming a specific table number, I locate the heading for the 2024 state table and grab the table that follows it.

This should make the collection step a little less fragile if the webpage changes its order later.

In [3]:
stats_url = "https://humantraffickinghotline.org/en/statistics"

response = requests.get(
    stats_url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")


year_heading = soup.find(
    lambda tag:
        tag.name in {"h2", "h3", "h4", "h5"}
        and "Signals and cases by State in 2024"
        in tag.get_text(" ", strip=True)
)


if year_heading is None:
    raise ValueError("I couldn't find the 2024 state statistics heading.")


state_table_html = year_heading.find_next("table")

if state_table_html is None:
    raise ValueError("The 2024 heading was found, but its table was not.")


hotline_raw = pd.read_html(
    StringIO(str(state_table_html))
)[0]


print(f"Rows collected: {len(hotline_raw)}")
print(f"Columns collected: {len(hotline_raw.columns)}")

hotline_raw.head(10)

Rows collected: 55
Columns collected: 5


,State,Signals received,% of total signals,Cases identified,% of total cases
0,Alabama,231,0.71%,105,0.87%
1,Alaska,46,0.14%,22,0.18%
2,Arizona,550,1.70%,300,2.50%
3,Arkansas,170,0.53%,80,0.67%
4,California,3378,10.46%,1734,14.45%
5,Colorado,465,1.44%,185,1.54%
6,Connecticut,167,0.52%,96,0.80%
7,Delaware,51,0.16%,32,0.27%
8,District of Columbia,135,0.42%,55,0.46%
9,Florida,1830,5.66%,832,6.93%


## First look at the data

Before changing anything, I want to confirm that the table looks like what I expected and preserve a copy of the original extract.

The raw version stays untouched in the project so I can always trace later transformations back to the source.

In [4]:
hotline_raw

,State,Signals received,% of total signals,Cases identified,% of total cases
0,Alabama,231,0.71%,105,0.87%
1,Alaska,46,0.14%,22,0.18%
2,Arizona,550,1.70%,300,2.50%
3,Arkansas,170,0.53%,80,0.67%
4,California,3378,10.46%,1734,14.45%
5,Colorado,465,1.44%,185,1.54%
6,Connecticut,167,0.52%,96,0.80%
7,Delaware,51,0.16%,32,0.27%
8,District of Columbia,135,0.42%,55,0.46%
9,Florida,1830,5.66%,832,6.93%


In [5]:
raw_file = (
    raw_data_dir
    / "national_hotline_state_statistics_2024_raw.csv"
)

hotline_raw.to_csv(raw_file, index=False)

print(f"Saved raw extract to:\n{raw_file}")

Saved raw extract to:
c:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis\data\raw\hotline\national_hotline_state_statistics_2024_raw.csv


## Creating a working copy

For a quick validation, I'm making a separate working copy with simpler column names and numeric fields.

The full cleaning and geographic preparation will happen in the next notebook.

In [6]:
hotline_2024 = hotline_raw.copy()

hotline_2024.columns = [
    "state",
    "signals_received",
    "pct_total_signals",
    "cases_identified",
    "pct_total_cases"
]


hotline_2024["state"] = (
    hotline_2024["state"]
    .astype("string")
    .str.strip()
)


for column in ["signals_received", "cases_identified"]:
    hotline_2024[column] = pd.to_numeric(
        hotline_2024[column]
        .astype(str)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )


for column in ["pct_total_signals", "pct_total_cases"]:
    hotline_2024[column] = pd.to_numeric(
        hotline_2024[column]
        .astype(str)
        .str.replace("%", "", regex=False),
        errors="coerce"
    )


hotline_2024["year"] = 2024
hotline_2024["source"] = "National Human Trafficking Hotline"


hotline_2024.head()

,state,signals_received,pct_total_signals,cases_identified,pct_total_cases,year,source
0,Alabama,231,0.71,105,0.87,2024,National Human Trafficking Hotline
1,Alaska,46,0.14,22,0.18,2024,National Human Trafficking Hotline
2,Arizona,550,1.70,300,2.50,2024,National Human Trafficking Hotline
3,Arkansas,170,0.53,80,0.67,2024,National Human Trafficking Hotline
4,California,3378,10.46,1734,14.45,2024,National Human Trafficking Hotline


## Basic quality check

At this stage I'm only checking for obvious problems like missing values, duplicated jurisdictions, and whether the state-level totals look reasonable.

The state table does not necessarily add up to the Hotline's national totals because not every reported signal or case has to be associated with one of these state-level rows.

In [7]:
quality_check = pd.DataFrame({
    "check": [
        "rows",
        "unique jurisdictions",
        "duplicate jurisdictions",
        "missing states",
        "missing signal counts",
        "missing case counts"
    ],
    "result": [
        len(hotline_2024),
        hotline_2024["state"].nunique(),
        hotline_2024["state"].duplicated().sum(),
        hotline_2024["state"].isna().sum(),
        hotline_2024["signals_received"].isna().sum(),
        hotline_2024["cases_identified"].isna().sum()
    ]
})

quality_check

,check,result
0,rows,55
1,unique jurisdictions,55
2,duplicate jurisdictions,0
3,missing states,0
4,missing signal counts,0
5,missing case counts,0


In [8]:
state_level_totals = {
    "signals_received": int(hotline_2024["signals_received"].sum()),
    "cases_identified": int(hotline_2024["cases_identified"].sum())
}

state_level_totals

{'signals_received': 23522, 'cases_identified': 11188}

## Which jurisdictions have the highest raw case counts?

This is only an initial look. Raw totals favor more populated states, so I don't want to draw conclusions from this ranking yet.

Later I will join population data and calculate reporting rates per 100,000 residents.

In [9]:
top_reported_cases = (
    hotline_2024[
        ["state", "signals_received", "cases_identified"]
    ]
    .sort_values("cases_identified", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top_reported_cases

,state,signals_received,cases_identified
0,California,3378,1734
1,Texas,2418,1360
2,Florida,1830,832
3,New York,1191,570
4,Illinois,792,385
5,Georgia,876,342
6,Michigan,764,340
7,Ohio,671,334
8,North Carolina,638,301
9,Arizona,550,300
